# Peer Grouping
- Unconstrained outlets within the same peer group is our reference ceiling.

In [1]:
# import neccessary libraries
import pandas as pd
import numpy as np

In [7]:
# Load the dataset
from pathlib import Path

# Path.cwd() gets the 'modeling' directory, and .parent moves up to the project root
project_root = Path.cwd().parent

# The division operator (/) is used by pathlib to join paths safely on any OS
outlets_stats_path = project_root / "lakehouse" / "silver" / "outlet_stats.parquet"
gold_path = project_root / "lakehouse" / "gold" / "master_features.parquet"

# Read the parquet file into a DataFrame
outlets_stats = pd.read_parquet(outlets_stats_path)
gold = pd.read_parquet(gold_path)

print("Datasets loaded successfully:")

Datasets loaded successfully:


In [8]:
# merge outlet stats with master features
gold = gold.merge(outlets_stats, on='Outlet_ID', how='left')

In [9]:
gold.info()

<class 'pandas.DataFrame'>
RangeIndex: 2334830 entries, 0 to 2334829
Data columns (total 33 columns):
 #   Column                 Dtype   
---  ------                 -----   
 0   Outlet_ID              str     
 1   Year                   int16   
 2   Month                  int8    
 3   Distributor_ID         str     
 4   SKU_ID                 str     
 5   Volume_Liters          float32 
 6   Total_Bill_Value       float64 
 7   Outlet_Size            category
 8   Cooler_Count           float32 
 9   Outlet_Type            category
 10  Latitude               float32 
 11  Longitude              float32 
 12  Seasonality_Index      category
 13  Holiday_Count          int64   
 14  poi_bus_stop_1500m     int8    
 15  poi_hospital_1500m     int8    
 16  poi_market_1500m       int8    
 17  poi_school_1500m       int8    
 18  poi_supermarket_1500m  int8    
 19  poi_tourism_1500m      int16   
 20  mean_vol               float64 
 21  max_vol                float64 
 22  min_v

In [10]:
gold.columns

Index(['Outlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID',
       'Volume_Liters', 'Total_Bill_Value', 'Outlet_Size', 'Cooler_Count',
       'Outlet_Type', 'Latitude', 'Longitude', 'Seasonality_Index',
       'Holiday_Count', 'poi_bus_stop_1500m', 'poi_hospital_1500m',
       'poi_market_1500m', 'poi_school_1500m', 'poi_supermarket_1500m',
       'poi_tourism_1500m', 'mean_vol', 'max_vol', 'min_vol', 'std_vol',
       'n_months', 'p90_vol', 'p10_vol', 'CV', 'utilization',
       'Ceiling_Hitting_Rate', 'Zero_Month_Rate', 'Constraint_Score',
       'is_constrained'],
      dtype='str')

In [ ]:
# Peer grouping

gold['Peer_Group'] = (str(gold['Outlet_Type']) + '_' + gold['Distributor_ID'])

# Compute Peer Ceiling
# Only from unconstrained outlets within each peer group
# These represent what's achievable without system constraints

peer_ceiling = gold[~gold['is_constrained']].groupby('Peer_Group').agg(
    peer_p90_unconstrained = ('max_vol', lambda x: x.quantile(0.90)),
    peer_median_unconstrained = ('max_vol', 'median'),
    peer_n_unconstrained = ('Outlet_ID', 'count'),
).reset_index()

In [16]:
# Save the updated gold dataframe
gold.to_parquet("../lakehouse/gold/gold_with_peers.parquet", index=False)